In [ ]:
# CONFIGURATION - change only this cell for a different site
from dotenv import load_dotenv
import os

load_dotenv()

API_KEY = os.getenv("NRL_API_KEY")
EMAIL = os.getenv("EMAIL")

LAT = 46.69115
LONG = -100.83192
YEARS = list(range(2015,2025))
INTERVAL = 60
ATTRIBUTES = (
    "ghi,dni,dhi,air_temperature,relative_humidity,"
    "dew_point,wind_speed,wind_direction,surface_albedo,solar_zenith_angle"
)
OUTPUT_PATH = "data/nsrdb_raw.csv"

In [ ]:
import requests
import pandas as pd
import os
import time
from io import StringIO

os.makedirs("data", exist_ok=True)

BASE_URL = "https://developer.nlr.gov/api/nsrdb/v2/solar/nsrdb-GOES-aggregated-v4-0-0-download.csv"

frames = []

for year in YEARS:
    params = {
        "api_key": API_KEY,
        "wkt": f"POINT({LONG} {LAT})",
        "names": year,
        "interval": INTERVAL,
        "attributes": ATTRIBUTES,
        "email": EMAIL,
        "utc": "false",
        "leap_day": "true",
        "full_name": "Student",
        "affiliation": "WGU",
        "reason": "Academic research",
        "mailing_list": "false",
    }

    print(f"Fetching {year} >>>", end=" ")
    resp = requests.get(BASE_URL, params=params, timeout=120)

    if resp.status_code != 200:
        print(f"ERROR {resp.status_code}: {resp.text[:200]}")
        continue

    # NSRDB returns 2 header rows before the column names row
    raw = StringIO(resp.text)
    df = pd.read_csv(raw, skiprows=2)
    df["Year"]=year
    frames.append(df)
    print(f"{len(df)} rows")
    time.sleep(10)

# Combine all years
full = pd.concat(frames, ignore_index=True)

# Build a proper datetime index
full["datetime"] = pd.to_datetime(
    full[["Year", "Month", "Day", "Hour", "Minute"]].rename(
        columns={
        "Year": "year",
        "Month": "month",
        "Day": "day",
        "Hour": "hour",
        "Minute": "minute"
        }
    )
)
full = full.set_index("datetime").sort_index()

# Drop the now redundant calendar columns
full.drop(columns=["Year", "Month", "Day", "Hour", "Minute"], inplace=True, errors="ignore")

print(f"\nShape: {full.shape}")
print(f"Date range: {full.index.min()} > {full.index.max()}")
print(f"Missing %: \n{(full.isnull().mean()*100).round(2)}")

full.to_csv(OUTPUT_PATH)
print(f"\nSaved to {OUTPUT_PATH}")